# vbpipe — bootstrap-train ball detector\nT4 GPU -> Run all. Cell 2 wants latest `vbpipe.zip` + `game_results.zip`. Total ~45-60 min.

In [ ]:
# Cell 1 — deps
!nvidia-smi -L
!pip -q install ultralytics

In [ ]:
# Cell 2 — upload vbpipe.zip (latest) and game_results.zip
from google.colab import files
up = files.upload()
import zipfile, os
for f in up:
    if "vbpipe" in f: zipfile.ZipFile(f).extractall("pipeline")
    else:
        os.makedirs("out", exist_ok=True); zipfile.ZipFile(f).extractall("out")
!pip -q install -e pipeline

In [ ]:
# Cell 3 — video from Drive
from google.colab import drive
drive.mount("/content/drive")
VIDEO = "/content/drive/MyDrive/example.mp4"  # <-- edit if needed
import os; print(os.path.exists(VIDEO))

In [ ]:
# Cell 4 — build dataset from physics-verified arcs + train (~30-40 min)
import sys; sys.path.insert(0, "pipeline")
from vbpipe.balltrain import build_dataset, train
data = build_dataset(VIDEO, "pipeline/ball_v3.json")
model = train(data)

In [ ]:
# Cell 5 — re-detect ball in all game rallies with trained model
import json
from vbpipe.balltrain import detect_all
g = json.load(open("out/game.json"))
for ri, r in enumerate(g["rallies"]):
    n = len({tr["id"] for tr in g["tracklets"] if tr["rally"]==ri and len(tr["boxes"])>=5})
    r["phase"] = "game" if (r["start"] >= 135 and n >= 8) else "warmup"
ball = detect_all(model, VIDEO, g["rallies"])
json.dump(ball, open("out/ball_trained.json","w"))

In [ ]:
# Cell 6 — contacts + plays + eval clips + download
from vbpipe.plays import find_contacts, attribute, classify
from vbpipe.annotate import render_rally
from collections import Counter
allp = []
for ri, (r, pts) in enumerate(zip(g["rallies"], ball)):
    if r.get("phase") != "game": r["contacts"] = []; continue
    cs = classify(attribute(find_contacts(pts), g["tracklets"], ri))
    r["contacts"] = cs; allp += [c["play"] for c in cs]
print("plays:", dict(Counter(allp)))
for r in g["rallies"]:
    if r.get("phase") != "game": continue
    print(f'  {r["start"]:6.0f}-{r["end"]:6.0f}  ' +
          " ".join(c["play"] for c in r["contacts"]))
json.dump(g, open("out/game.json","w"), indent=1)
import os; os.makedirs("out/eval", exist_ok=True)
order = sorted([i for i,r in enumerate(g["rallies"]) if r.get("phase")=="game"],
               key=lambda i: -len(g["rallies"][i]["contacts"]))
for i in order[:4]:
    render_rally(VIDEO, g["rallies"][i], ball[i], g["rallies"][i]["contacts"],
                 f"out/eval/rally_{i:02d}.mp4")
!cd out && zip -q -r ../trained_results.zip game.json ball_trained.json eval
from google.colab import files
files.download("trained_results.zip")